In [0]:
import sys
sys.path.append("..")

from ingestion.utils.config_manager import ConfigManager, SOURCE_SYSTEM_TABLE, CONFIG_MASTER_TABLE
from ingestion.utils.secrets import SecretResolver

In [0]:

# 1. Retrieve the source_name passed from the Databricks Job
dbutils.widgets.text("source_system_id", "1")
source_id = dbutils.widgets.get("source_system_id").strip()

print(f"Fetching configuration for source: {source_id}...")

# 2. Query the Unity Catalog configuration table for Data Dictionary config
query = f"""
    SELECT 
        source_id,
        source_name,
        source_type,
        host,
        port,
        database_name,
        driver_class,
        secret_scope,
        secret_key_credentials,
        source_id,
        Data_Dictionary_Enable,
        landing_volume_path
    FROM migration_x_catalog.pfl_x_schema.config_source_system 
    WHERE source_id = int('{source_id}') and is_active = 1;
"""
config_df = spark.sql(query).collect()

if not config_df:
    print(f"WARNING: No configuration found for source '{source_name}'. Exiting.")
    dbutils.jobs.taskValues.set(key="run_data_dict", value="false")
    dbutils.notebook.exit(f"No config found for source: {source_name}")

source_dict = config_df[0]

# 3. Extract Data Dictionary parameters
dd_enabled = int(source_dict["Data_Dictionary_Enable"] or 0)
s3_bucket = str(source_dict["landing_volume_path"]).strip() if source_dict["landing_volume_path"] else ""
source_name = str(source_dict["source_name"]).strip() if source_dict["source_name"] else ""

# Push task values for downstream workflow tasks
run_data_dict = "true" if dd_enabled == 1 else "false"
dbutils.jobs.taskValues.set(key="run_data_dict", value=run_data_dict)
dbutils.jobs.taskValues.set(key="landing_volume_path", value=s3_bucket)

print(f"Data_Dictionary_Enable = {dd_enabled} | run_data_dict = {run_data_dict}")

In [0]:
# DATA DICTIONARY EXECUTION FLOW
# Only runs when Data_Dictionary_Enable = 1

if dd_enabled != 1:
    print("Data Dictionary is DISABLED for this source. Skipping.")
    dbutils.notebook.exit("SKIPPED: Data Dictionary not enabled.")

print(f"Data Dictionary is ENABLED. Starting flow for source: {source_name}")
nb_path = f"/Workspace/Shared/pfl-ingestion-framework-dd-check/databricks-ingestion-framework/src/dd_validation/get_all_table_data_dictionary"
# TODO - Create the data_dictionary folder and data_dictionary file inside the PFL/Admin/Config/Data_Dictionary/get_all_table_data_dictionary

# First check whether the data_dictionary folder is there or not. If it is not there then create it.
dd_folder_name = "data_dictionary"
dd_folder_path = f"{s3_bucket}/{dd_folder_name}"
dd_file_name = source_name + "_source_data_dictionary.parquet"

raw_s3_path = f"{s3_bucket}/{dd_folder_name}/{dd_file_name}"

# Validate required config values
if not nb_path:
    raise ValueError("notebook_path is empty in config. Cannot determine which notebook returns the DD query.")
if not s3_bucket or not dd_folder_name or not dd_file_name:
    raise ValueError("S3 destination config incomplete. Check Raw_S3_Bucket_Name, Raw_DD_Folder, Raw_DD_File_Name.")

# Construct the target S3 path
raw_s3_path = f"{s3_bucket}/{dd_folder_name}/{dd_file_name}"
print(f"Target S3 path: {raw_s3_path}")


# STEP 1: Run the configured notebook to get the Data Dictionary query
print(f"\nStep 1: Running notebook '{nb_path}' to fetch extraction query...")

source_query = dbutils.notebook.run(
    path=nb_path,
    timeout_seconds=600,
    arguments={"source_name": source_name, "code_part": "1"}
)

if not source_query or not source_query.strip():
    raise ValueError(f"Notebook '{nb_path}' did not return a valid query.")

print(f"Query retrieved successfully:\n{source_query}")

In [0]:
print(s3_bucket[5:-1])
print(dd_file_name[:-8])

In [0]:
# STEP 2: Connect to source system & execute the query using existing framework
print(f"\nStep 2: Connecting to source system and executing query...")

# Resolve credentials from the source system config
secrets = SecretResolver(dbutils)
username, password = secrets.get_credentials(
    source_dict["secret_scope"],
    source_dict["secret_key_credentials"]
)

# Build JDBC connection options using the framework's URL builder pattern
from ingestion.connectors.jdbc_connector import _build_url, _DEFAULT_DRIVER

source_type = str(source_dict["source_type"]).upper()
host = source_dict["host"]
port = int(source_dict["port"]) if source_dict["port"] else 0
database_name = source_dict["database_name"] or ""

jdbc_url = _build_url(
    source_type=source_type,
    host=host,
    port=port,
    database_name=database_name,
    extra_params={}
)

driver = source_dict["driver_class"] if source_dict["driver_class"] else _DEFAULT_DRIVER.get(source_type)
if not driver:
    raise ValueError(f"No JDBC driver found for source_type='{source_type}'. Set driver_class in config.")

print(f"JDBC URL: {jdbc_url}")
print(f"Driver: {driver}")

# Execute the Data Dictionary query against the source system
try:
    raw_df = (spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("user", username)
        .option("password", password)
        .option("driver", driver)
        .option("query", dict_query)
        .load())

    row_count = raw_df.count()
    print(f"Query returned {row_count} rows.")

except Exception as e:
    raise RuntimeError(f"JDBC extraction failed for source '{source_name}': {str(e)}")


# STEP 2.1: Write results to the configured S3 location
print(f"\nStep 3: Writing {row_count} rows to {raw_s3_path}...")

try:
    raw_df.write.format("parquet").mode("overwrite").save(raw_s3_path)
    print(f"Data Dictionary data successfully written to: {raw_s3_path}")
except Exception as e:
    raise RuntimeError(f"Failed to write Data Dictionary to S3: {str(e)}")


In [0]:
# STEP 4: Re-run the configured notebook with code_part=2 (post-extraction)
print(f"\nStep 4: Running notebook '{nb_path}' with code_part=2 (post-extraction)...")

try:
    result_part2 = dbutils.notebook.run(
        path=nb_path,
        timeout_seconds=600,
        arguments={"code_part": "2", "s3_bucket": s3_bucket[5:-1], "dd_view_name": dd_file_name[:-8]}
    )
    print(f"Notebook code_part=2 completed. Result: {result_part2}")
except Exception as e:
    raise RuntimeError(f"Notebook '{nb_path}' code_part=2 failed: {str(e)}")

In [0]:
print(f"\n{'='*60}")
print(f"DATA DICTIONARY FLOW COMPLETED SUCCESSFULLY")
print(f"Source: {source_name}")
print(f"Rows extracted: {row_count}")
print(f"Output: {raw_s3_path}")
print(f"{'='*60}")

dbutils.notebook.exit(f"SUCCESS: {row_count} rows written to {raw_s3_path}")

In [0]:
%sql
-- Template for updating a table
-- UPDATE migration_x_catalog.pfl_x_schema.config_source_system
-- SET
--     Data_Dictionary_Enable = 1
-- WHERE source_name = 'PG_TEST_RDS';

In [0]:
# %sql
# ALTER TABLE migration_x_catalog.pfl_x_schema.config_source_system
# DROP COLUMNS (
#     Raw_S3_Bucket_Name,
#     Raw_DD_Folder,
#     Raw_DD_File_Name,
#     notebook_path,
#     Trigger_Time
# );
